In [ ]:
%load_ext autoreload
%autoreload 2
# Import required libraries
import sys
sys.path.append("../")
import os
import numpy as np
import utm
import io
import shutil
import h5py
import json
import matplotlib.pyplot as plt
from PIL import Image
from typing import List
import cv2

# Load customized utility functions
from utils.transformations import rotation_matrix_from_angles, get_yaw_pitch_roll, filter_above_ground, compute_heading
from utils.plot import plot_lidar_camera_oxts
from utils.process import memory_usage

# Load ZOD DevKit
from zod import ZodFrames, ZodSequences, ZodDrives
import zod.constants as constants
from zod.constants import Camera, Lidar, Anonymization, AnnotationProject
from zod.data_classes.ego_motion import OXTS_TIMESTAMP_OFFSET, interpolate_transforms
from zod.data_classes import LidarData
from zod.utils.geometry import transform_points
from zod.visualization.lidar_on_image import get_3d_transform_camera_lidar
from zod.constants import Camera, Lidar, Anonymization

# Load aerial image download functions
test_country = 'PL'

if test_country == 'PL': # can be used as training 
    from Map_Downloading.scripts.poland import poland_exact_position_image as download_sat
    from Map_Downloading.scripts.poland import RESOLUTION as resolution
    center_meridian = 19
elif test_country == 'SE': # can be used as training 
    from Map_Downloading.scripts.sweden import sweden_exact_position_image as download_sat
    from Map_Downloading.scripts.sweden import RESOLUTION as resolution
    center_meridian = 15
elif test_country == 'IT': # localization of IT is very bad. Do not use it as training nor testing
    from Map_Downloading.scripts.Italy import Italy_exact_position_image as download_sat
    from Map_Downloading.scripts.Italy import resolutions
    zoom = 18
    resolution = resolutions.get(zoom)
    center_meridian = 15
elif test_country == 'FR': # can be used as training
    from Map_Downloading.scripts.france import france_exact_position_image as download_sat
    from Map_Downloading.scripts.france import TRUE_RESOLUTION as resolution
elif test_country == 'NO': # noticable error in orientation. Do not use it as training nor testing
    from Map_Downloading.scripts.norway import norway_exact_position_image as download_sat
    from Map_Downloading.scripts.norway import RESOLUTION as resolution
    center_meridian = 15
elif test_country == 'NL': # can be used as training 
    from Map_Downloading.scripts.netherlands import nl_exact_position_image as download_sat
    from Map_Downloading.scripts.netherlands import RESOLUTION as resolution




# Set matplotlib settings
%matplotlib inline

# NOTE! Set the path to dataset and choose a version
# dataset_root = "/Users/ziminxia/Work/Collaborations/Zenseact/data"  # your local path to zod
dataset_root = "/mnt/data/Work/datasets/zod"  # your local path to zod
version = "full"  # "mini" or "full"


# Load and display sensor frames
sensor_frames = plt.imread('./sensor_frames.png')
plt.figure(figsize=(20, 10))
plt.imshow(sensor_frames)
plt.axis('off')
plt.show()


In [ ]:
def approximate_meridian_convergence(lat_deg, lon_deg, central_meridian_deg):
    """
    Approximates the meridian convergence (in degrees) given a point's latitude and longitude,
    and the central meridian of the map projection.

    Args:
        lat_deg (float): Latitude in degrees.
        lon_deg (float): Longitude in degrees.
        central_meridian_deg (float): Central meridian longitude in degrees.

    Returns:
        float: Meridian convergence in degrees.
    """
    lat_rad = np.radians(lat_deg)
    lon_rad = np.radians(lon_deg)
    cm_rad = np.radians(central_meridian_deg)
    gamma_rad = np.arctan(np.tan(lon_rad - cm_rad) * np.sin(lat_rad))
    return np.degrees(gamma_rad)

In [ ]:
# initialize ZodFrames
zod_frames = ZodFrames(dataset_root=dataset_root, version=version)

# get default training and validation splits
training_frames = zod_frames.get_split(constants.TRAIN)
validation_frames = zod_frames.get_split(constants.VAL)

# print the number of training and validation frames
print(f"Number of training frames: {len(training_frames)}")
print(f"Number of validation frames: {len(validation_frames)}")

all_frame_num = list(training_frames) + list(validation_frames)
print(f"Number of total frames: {len(all_frame_num)}")

frames_per_country = {}

for i in range(len(all_frame_num)):
    zod_frame = zod_frames[all_frame_num[i]]
    metadata = zod_frame.metadata
    # print(f"Country Code: {metadata.country_code}")

    if metadata.country_code in frames_per_country:
        frames_per_country[metadata.country_code].append(all_frame_num[i])
    else:
        frames_per_country[metadata.country_code] = [all_frame_num[i]]

for country in frames_per_country.keys():
    print(country, len(frames_per_country[country]))

### Use oxts of the first frame and ego-motion to get current pose (not recommend)

In [ ]:
# # plot frames location per country
# # country_list = frames_per_country.keys()
# size = 100
# country_list = ['SE']  # Modify as needed

# print(country_list)

# for country in country_list:
#     print('country', country)
               
#     for frame_idx in frames_per_country[country]:
#         print('frame_idx', frame_idx)
#         frame = zod_frames[frame_idx]
#         calibrations = frame.calibration
#         cam_intrinsics_4x3 = calibrations.cameras[Camera.FRONT].intrinsics
#         cam_intrinsics = cam_intrinsics_4x3[:, :3]
#         cam_distortion = calibrations.cameras[Camera.FRONT].distortion
#         image_dimensions = tuple(calibrations.cameras[Camera.FRONT].image_dimensions)
        

#         # === Compute undistortion maps ===
#         K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
#             cam_intrinsics, cam_distortion, image_dimensions, np.eye(3), balance=0.0
#         )
        
#         map1, map2 = cv2.fisheye.initUndistortRectifyMap(
#             cam_intrinsics, cam_distortion, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2
#         )
    
#         T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
#         T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

#         # Save Ground Image
#         grd_image = frame.get_image(Anonymization.BLUR)
#         grd_image = cv2.remap(grd_image, map1, map2, interpolation=cv2.INTER_LINEAR)
#         # Image.fromarray(grd_image).resize((grd_image.shape[1] // 4, grd_image.shape[0] // 4), Image.LANCZOS).save(
#         #     os.path.join(ground_path, f'frame{i:06}.png'), format="PNG"
#         # )
#         plt.imshow(grd_image)
#         plt.axis('off')
#         plt.show()

#         frame_timestamp = frame.info.keyframe_time.timestamp()
#         current_pose = frame.oxts.get_poses(frame_timestamp)

#         core_lidar = frame.get_lidar()[0]
#         compensated_lidar = frame.compensate_lidar(core_lidar, frame_timestamp)
#         pcd = compensated_lidar.points

#         filename = f"{os.path.join(dataset_root, 'single_frames')}/{frame_idx}/oxts.hdf5"
        
#         with h5py.File(filename, "r") as f:
#             lat_oxts, lon_oxts, alt_oxts = f['posLat'][()][0], f['posLon'][()][0], f['posAlt'][()][0]
#             yaw_oxts, pitch_oxts, roll_oxts = 90-f['heading'][()][0], f['pitch'][()][0], f['roll'][()][0]

        
                
#         T_origin2utm = np.eye(4)
#         T_origin2utm[:3, :3] = rotation_matrix_from_angles(
#             np.radians(roll_oxts), np.radians(pitch_oxts), np.radians(yaw_oxts), order="ZYX"
#         )
        
#         easting_oxts, northing_oxts, zone_number, zone_letter = utm.from_latlon(lat_oxts, lon_oxts)
#         T_origin2utm[:3, 3] = [easting_oxts, northing_oxts, alt_oxts]

#         T_current2utm = T_origin2utm @ current_pose
#         # T_cam2utm = T_current2utm @ T_cam2oxts

#         pcd_utm = transform_points(pcd, T_current2utm @ T_lidar2oxts)
#         pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

       
#         # easting_frame = T_cam2utm[0,3]
#         # northing_frame = T_cam2utm[1,3]

#         yaw_current, _, _ = get_yaw_pitch_roll(T_current2utm)
#         heading_current = 90 - np.degrees(yaw_current)
        
#         # Retrieve Aerial Image
#         lat_current, lon_current = utm.to_latlon(T_current2utm[0, 3], T_current2utm[1, 3], zone_number, zone_letter)
#         if country == 'SE':
#             aerial_img = sweden_exact_position_image(lat_current, lon_current, size)
#             resolution = sweden_resolution
#         elif country == 'FR':
#             aerial_img = france_exact_position_image(lat_current, lon_current, size)
#             resolution = france_resolution

#         # Plot BEV (Bird’s Eye View) with LiDAR points
#         bev_image = np.array(aerial_img)
#         H, W = bev_image.shape[:2]
#         bev_center = (T_current2utm[0, 3], T_current2utm[1, 3])

#         # Convert LiDAR points to BEV pixel coordinates
#         x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
#         y_indices = H - ((pcd_utm[:, 1] - bev_center[1]) / resolution + H / 2).astype(int)
#         x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)

#         plt.figure(figsize=(8, 8))
#         plt.imshow(bev_image, origin='upper')
#         plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
#         plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_current)), np.cos(np.radians(heading_current)), color='r', scale=20, label='OXTS')
#         plt.legend(loc=2)
#         plt.axis('off')
#         # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
#         plt.show()
#         plt.close()

    

### Use previous and next oxts measure (global positioning) to get current pose (recommended)

In [ ]:
# plot frames location per country
# country_list = frames_per_country.keys()
size = 100

country_list = [test_country]  # Modify as needed

print(country_list)

for country in country_list:
    print('country', country)
               
    for frame_idx in frames_per_country[country]:
        print('frame_idx', frame_idx)
        frame = zod_frames[frame_idx]
        calibrations = frame.calibration
        cam_intrinsics_4x3 = calibrations.cameras[Camera.FRONT].intrinsics
        cam_intrinsics = cam_intrinsics_4x3[:, :3]
        cam_distortion = calibrations.cameras[Camera.FRONT].distortion
        image_dimensions = tuple(calibrations.cameras[Camera.FRONT].image_dimensions)
    
        K_new = cv2.fisheye.estimateNewCameraMatrixForUndistortRectify(
            cam_intrinsics, cam_distortion, image_dimensions, np.eye(3), balance=0.0
        )
        
        map1, map2 = cv2.fisheye.initUndistortRectifyMap(
            cam_intrinsics, cam_distortion, np.eye(3), K_new, image_dimensions, cv2.CV_16SC2
        )
        
        T_cam2oxts = calibrations.get_extrinsics(Camera.FRONT).transform
        T_lidar2oxts = calibrations.lidars[Lidar.VELODYNE].extrinsics.transform

        # Save Ground Image
        grd_image = frame.get_image(Anonymization.BLUR)
        grd_image = cv2.remap(grd_image, map1, map2, interpolation=cv2.INTER_LINEAR)
        # Image.fromarray(grd_image).resize((grd_image.shape[1] // 4, grd_image.shape[0] // 4), Image.LANCZOS).save(
        #     os.path.join(ground_path, f'frame{i:06}.png'), format="PNG"
        # )
        plt.imshow(grd_image)
        plt.axis('off')
        plt.show()

        frame_timestamp = frame.info.keyframe_time.timestamp()

        core_lidar = frame.get_lidar()[0]
        compensated_lidar = frame.compensate_lidar(core_lidar, frame_timestamp)
        pcd = compensated_lidar.points

        filename = f"{os.path.join(dataset_root, 'single_frames')}/{frame_idx}/oxts.hdf5"
        
        with h5py.File(filename, "r") as f:
            lat_oxts, lon_oxts, alt_oxts = f['posLat'][()], f['posLon'][()], f['posAlt'][()]
            yaw_oxts, pitch_oxts, roll_oxts = 90-f['heading'][()], f['pitch'][()], f['roll'][()]
            oxts_timestamp = OXTS_TIMESTAMP_OFFSET + f['timestamp'][()] + f['leapSeconds'][()][0]

        
        # Use np.searchsorted for efficient timestamp lookup
        oxts_idx = np.searchsorted(oxts_timestamp, frame_timestamp, side='right')
        
        # Extract previous and next OXTS readings
        lat_prev, lon_prev, alt_prev = lat_oxts[oxts_idx - 1], lon_oxts[oxts_idx - 1], alt_oxts[oxts_idx - 1]
        yaw_prev, pitch_prev, roll_prev = yaw_oxts[oxts_idx - 1], pitch_oxts[oxts_idx - 1], roll_oxts[oxts_idx - 1]
        easting_prev, northing_prev, zone_number_prev, zone_letter_prev = utm.from_latlon(lat_prev, lon_prev)

        lat_next, lon_next, alt_next = lat_oxts[oxts_idx], lon_oxts[oxts_idx], alt_oxts[oxts_idx]
        yaw_next, pitch_next, roll_next = yaw_oxts[oxts_idx], pitch_oxts[oxts_idx], roll_oxts[oxts_idx]
        easting_next, northing_next, zone_number_next, zone_letter_next = utm.from_latlon(lat_next, lon_next)

        if test_country == 'FR' or test_country == 'NL': # can be used as training 
            gamma_prev = 0
            gamma_next = 0 
        else:
            gamma_prev = approximate_meridian_convergence(lat_prev, lon_prev, center_meridian)
            gamma_next = approximate_meridian_convergence(lat_next, lon_next, center_meridian)

        yaw_prev += gamma_prev # heading should subtract gamma, hence yaw should be adding 
        yaw_next += gamma_next

        # Compute previous and next transformation matrices
        T_previous = np.eye(4)
        T_previous[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_prev), np.radians(pitch_prev), np.radians(yaw_prev), order="ZYX"
        )
        T_previous[:3, 3] = [easting_prev, northing_prev, alt_prev]

        T_next = np.eye(4)
        T_next[:3, :3] = rotation_matrix_from_angles(
            np.radians(roll_next), np.radians(pitch_next), np.radians(yaw_next), order="ZYX"
        )
        T_next[:3, 3] = [easting_next, northing_next, alt_next]

        # Interpolate transformation
        interp_factor = (frame_timestamp - oxts_timestamp[oxts_idx - 1]) / (oxts_timestamp[oxts_idx] - oxts_timestamp[oxts_idx - 1])
        T_current2utm = interpolate_transforms(T_previous, T_next, interp_factor)
        yaw_current, _, _ = get_yaw_pitch_roll(T_current2utm) # 0 is East, 90 is North
        heading_current = 90 - np.degrees(yaw_current) # 0 is North, 90 is East
        

        T_cam2utm = T_current2utm @ T_cam2oxts
        
        print('yaw_current', np.degrees(yaw_current) )

        R_cam2utm = T_cam2utm[:3, :3]
        f_cam = np.array([0, 0, 1])  # camera's forward axis in its local frame
        f_world = R_cam2utm @ f_cam  # camera's forward axis in world frame
        
        # horizontal projection (ignore height)
        f_world_xy = f_world[:2] / np.linalg.norm(f_world[:2])
        
        heading_from_forward = np.degrees(np.arctan2(f_world_xy[1], f_world_xy[0]))
        print("Heading from forward vector:", heading_from_forward)

        pcd_utm = transform_points(pcd, T_current2utm @ T_lidar2oxts)
        # pcd_utm = filter_above_ground(pcd_utm, ground_threshold=1)

       
        # easting_frame = T_cam2utm[0,3]
        # northing_frame = T_cam2utm[1,3]

        
        
        # Retrieve Aerial Image
        lat_current, lon_current = utm.to_latlon(T_current2utm[0, 3], T_current2utm[1, 3], zone_number_prev, zone_letter_prev)
        aerial_img = download_sat(lat_current, lon_current, size)

        # Plot BEV (Bird’s Eye View) with LiDAR points
        bev_image = np.array(aerial_img)
        H, W = bev_image.shape[:2]
        print('H, W', H, W)
        bev_center = (T_current2utm[0, 3], T_current2utm[1, 3])

        # Convert LiDAR points to BEV pixel coordinates
        x_indices = ((pcd_utm[:, 0] - bev_center[0]) / resolution + W / 2).astype(int)
        y_indices = H - ((pcd_utm[:, 1] - bev_center[1]) / resolution + H / 2).astype(int)
        x_indices, y_indices = np.clip(x_indices, 0, W - 1), np.clip(y_indices, 0, H - 1)

        plt.figure(figsize=(8, 8))
        plt.imshow(bev_image, origin='upper')
        plt.scatter(x_indices, y_indices, s=0.1, c='purple', alpha=0.3, label="LiDAR Points")
        plt.quiver(W / 2, H / 2, np.sin(np.radians(heading_current)), np.cos(np.radians(heading_current)), color='r', scale=20, label='OXTS')
        plt.legend(loc=2)
        plt.axis('off')
        # plt.savefig(os.path.join(aerial_path, f'frame{i:06}.png'), bbox_inches='tight')
        plt.show()
        plt.close()

    

In [ ]:
lat_current, lon_current = utm.to_latlon(T_current2utm[0, 3], T_current2utm[1, 3], zone_number_prev, zone_letter_prev)
aerial_img = download_sat(lat_current, lon_current, size)
bev_image = np.array(aerial_img)
H, W = bev_image.shape[:2]
print('H, W', H, W)
bev_center = (T_current2utm[0, 3], T_current2utm[1, 3])



plt.figure(figsize=(8, 8))
plt.imshow(bev_image, origin='upper')

lat_current, lon_current = utm.to_latlon(T_current2utm[0, 3], T_current2utm[1, 3]-50, zone_number_prev, zone_letter_prev)
aerial_img = download_sat(lat_current, lon_current, size)
bev_image = np.array(aerial_img)
H, W = bev_image.shape[:2]
print('H, W', H, W)
bev_center = (T_current2utm[0, 3], T_current2utm[1, 3])



plt.figure(figsize=(8, 8))
plt.imshow(bev_image, origin='upper')
